# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv, find_dotenv


In [3]:
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"
assert os.getenv("CHROMA_DB_PATH"), "CHROMA_DB_PATH is not set"


### VectorDB Instance

In [4]:
# Persist the vector DB at the path from .env (checked by the assert above)
chroma_client = chromadb.PersistentClient(path=os.environ["CHROMA_DB_PATH"])

### Collection

In [5]:
from typing import cast
from chromadb.api.types import Embeddable, EmbeddingFunction

embedding_fn = cast(EmbeddingFunction[Embeddable], embedding_functions.OpenAIEmbeddingFunction())


In [6]:
# get_or_create so re-running reuses the persisted collection instead of erroring
collection = chroma_client.get_or_create_collection(
   name="udaplay",
   embedding_function=embedding_fn
)

### Add documents

In [7]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    # upsert so re-running refreshes existing entries instead of silently skipping them
    collection.upsert(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

### Query the collection

A direct semantic search against ChromaDB, with no LLM generating an answer: the query text is embedded and compared against the stored game vectors, and the raw matches are printed.

In [8]:
query = "Super Mario 64"

results = collection.query(
    query_texts=[query],
    n_results=3,
    include=["documents", "metadatas", "distances"],
)

ids = results["ids"][0]
documents = (results["documents"] or [[]])[0]
metadatas = (results["metadatas"] or [[]])[0]
distances = (results["distances"] or [[]])[0]

print(f"Query: {query!r} -> {len(ids)} result(s), closest first (lower distance = more similar)\n")
for rank, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances), start=1):
    print(f"#{rank}  id={doc_id}  distance={distance:.4f}")
    print(f"    document: {document}")
    print(f"    metadata: {json.dumps(dict(metadata), ensure_ascii=False)}\n")

print("-"*80)
print("\nQuerying the collection directly for the same game (should return the same results):\n")
print(collection.get(where_document={"$contains": "Mario"}, include=["documents", "metadatas"]))

Query: 'Super Mario 64' -> 3 result(s), closest first (lower distance = more similar)

#1  id=009  distance=0.0828
    document: [Nintendo 64] Super Mario 64 (1996) - A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.
    metadata: {"Platform": "Nintendo 64", "Name": "Super Mario 64", "Description": "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.", "Publisher": "Nintendo", "Genre": "Platformer", "YearOfRelease": 1996}

#2  id=008  distance=0.1274
    document: [Super Nintendo Entertainment System (SNES)] Super Mario World (1990) - A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.
    metadata: {"Publisher": "Nintendo", "Description": "A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.", "Platform": "Super Nintendo Entertainment Sys